In [ ]:
import numpy as np
import os
import gc
import joblib
from tqdm.auto import tqdm
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [ ]:
# --- 1. CONFIGURACIÓN ---
DIR_BASE = 'D:/Proyecto_Embeddings'

# Archivos de Features (X)
FILE_X_TRAIN = os.path.join(DIR_BASE, 'dataset_train_completo.dat')
FILE_X_EVAL  = os.path.join(DIR_BASE, 'dataset_eval_completo.dat')

# Archivos de Targets (y) - Nombres basados en tu foto
FILES_Y = {
    'train': {
        'cap': os.path.join(DIR_BASE, 'y_train_cap.npy'),
        'ini': os.path.join(DIR_BASE, 'y_train_ini.npy'),
        'fin': os.path.join(DIR_BASE, 'y_train_fin.npy')
    },
    'eval': {
        'cap': os.path.join(DIR_BASE, 'y_eval_cap.npy'),
        'ini': os.path.join(DIR_BASE, 'y_eval_ini.npy'),
        'fin': os.path.join(DIR_BASE, 'y_eval_fin.npy')
    }
}

COLS = 772          # 768 BERT + 4 Manuales
DTYPE = np.float16  # Formato en disco

# --- 2. CONEXIÓN A DATOS ---
def get_rows(filepath, cols, dtype):
    return os.path.getsize(filepath) // (cols * np.dtype(dtype).itemsize)

print("🔌 Conectando a disco...")
n_train = get_rows(FILE_X_TRAIN, COLS, DTYPE)
n_eval  = get_rows(FILE_X_EVAL, COLS, DTYPE)

# Mapeamos X (Lectura desde disco)
X_train = np.memmap(FILE_X_TRAIN, dtype=DTYPE, mode='r', shape=(n_train, COLS))
X_eval  = np.memmap(FILE_X_EVAL,  dtype=DTYPE, mode='r', shape=(n_eval, COLS))

# Cargamos Y en RAM (Diccionarios para acceso fácil)
print("📥 Cargando etiquetas en RAM...")
y_train = {
    'capitalizacion': np.load(FILES_Y['train']['cap']),
    'punt_inicial':   np.load(FILES_Y['train']['ini']),
    'punt_final':     np.load(FILES_Y['train']['fin'])
}
y_eval = {
    'capitalizacion': np.load(FILES_Y['eval']['cap']),
    'punt_inicial':   np.load(FILES_Y['eval']['ini']),
    'punt_final':     np.load(FILES_Y['eval']['fin'])
}

print(f"✅ Datos listos. Train: {n_train:,} filas | Eval: {n_eval:,} filas")

# --- 3. CONFIGURACIÓN DE LOS 3 MODELOS SVM ---
print("\n🚀 Configurando SVMs Lineales...")

# loss='hinge' es lo que lo convierte en un SVM.
# alpha=0.0001 es la regularización estándar.
modelos = {
    'capitalizacion': SGDClassifier(loss='hinge', penalty='l2', alpha=1e-4, n_jobs=1, random_state=42),
    'punt_inicial':   SGDClassifier(loss='hinge', penalty='l2', alpha=1e-4, n_jobs=1, random_state=42),
    'punt_final':     SGDClassifier(loss='hinge', penalty='l2', alpha=1e-4, n_jobs=1, random_state=42)
}

# Detectar clases automáticamente (Necesario para el entrenamiento incremental)
clases_unicas = {}
for nombre, targets in y_train.items():
    clases_unicas[nombre] = np.unique(targets)
    print(f"   Target '{nombre}' tiene clases: {clases_unicas[nombre]}")

# Scaler (Obligatorio para SVM)
scaler = StandardScaler()

# --- 4. ENTRENAMIENTO (SINGLE PASS) ---
BATCH_SIZE = 50000
print("\n🔥 INICIANDO ENTRENAMIENTO (Batch Learning)...")

# FASE A: Ajustar Scaler (Calcula media y desviación)
print("   [1/2] Ajustando Scaler...")
for i in tqdm(range(0, n_train, BATCH_SIZE), desc="Scaling"):
    end = min(i + BATCH_SIZE, n_train)
    X_batch = X_train[i:end].astype(np.float32)
    scaler.partial_fit(X_batch)

# FASE B: Entrenar Modelos
print("   [2/2] Entrenando los 3 modelos simultáneamente...")
# Barajamos índices para mejorar convergencia del gradiente
indices = np.random.permutation(n_train)

for i in tqdm(range(0, n_train, BATCH_SIZE), desc="Training"):
    # 1. Cargar Batch X
    idx_batch = indices[i : i + BATCH_SIZE]
    X_batch = X_train[idx_batch].astype(np.float32)
    X_batch = scaler.transform(X_batch) # Normalizar

    # 2. Entrenar cada modelo con SU target
    for nombre, clf in modelos.items():
        y_batch_target = y_train[nombre][idx_batch]
        clf.partial_fit(X_batch, y_batch_target, classes=clases_unicas[nombre])

print("✅ ¡Modelos Entrenados!")

# --- 5. GUARDAR MODELOS (Checkpoint) ---
# Guardamos para no tener que re-entrenar
os.makedirs('modelos_entrenados', exist_ok=True)
joblib.dump(scaler, 'modelos_entrenados/scaler.pkl')
for nombre, clf in modelos.items():
    joblib.dump(clf, f'modelos_entrenados/svm_{nombre}.pkl')
print("💾 Modelos guardados en carpeta 'modelos_entrenados/'")

# --- 6. EVALUACIÓN FINAL ---
print("\n📊 Evaluando en Set de Validación...")

# Diccionario para acumular predicciones
preds = {k: [] for k in modelos.keys()}

# Iteramos X_eval (Batch a Batch para no saturar RAM al predecir)
for i in tqdm(range(0, n_eval, BATCH_SIZE), desc="Predicting"):
    end = min(i + BATCH_SIZE, n_eval)
    X_batch = X_eval[i:end].astype(np.float32)
    X_batch = scaler.transform(X_batch)

    for nombre, clf in modelos.items():
        batch_p = clf.predict(X_batch)
        preds[nombre].extend(batch_p)

# Reportes
for nombre in modelos.keys():
    print(f"\n{'='*40}")
    print(f" REPORTE: {nombre.upper()}")
    print(f"{'='*40}")
    print(classification_report(y_eval[nombre], preds[nombre]))

🔌 Conectando a disco...
📥 Cargando etiquetas en RAM...
✅ Datos listos. Train: 7,046,323 filas | Eval: 780,666 filas

🚀 Configurando SVMs Lineales...
   Target 'capitalizacion' tiene clases: [0 1 2 3]
   Target 'punt_inicial' tiene clases: ['' '¿']
   Target 'punt_final' tiene clases: ['' ',' '.' '?']

🔥 INICIANDO ENTRENAMIENTO (Batch Learning)...
   [1/2] Ajustando Scaler...


Scaling:   0%|          | 0/141 [00:00<?, ?it/s]

   [2/2] Entrenando los 3 modelos simultáneamente...


Training:   0%|          | 0/141 [00:00<?, ?it/s]

✅ ¡Modelos Entrenados!
💾 Modelos guardados en carpeta 'modelos_entrenados/'

📊 Evaluando en Set de Validación...


Predicting:   0%|          | 0/16 [00:00<?, ?it/s]


 REPORTE: CAPITALIZACION
              precision    recall  f1-score   support

           0       0.88      0.98      0.93    669579
           1       0.57      0.20      0.30    104422
           2       0.97      0.69      0.81      4399
           3       0.46      0.18      0.26      2266

    accuracy                           0.87    780666
   macro avg       0.72      0.51      0.57    780666
weighted avg       0.84      0.87      0.84    780666


 REPORTE: PUNT_INICIAL
              precision    recall  f1-score   support

                   0.99      1.00      1.00    773959
           ¿       0.51      0.04      0.08      6707

    accuracy                           0.99    780666
   macro avg       0.75      0.52      0.54    780666
weighted avg       0.99      0.99      0.99    780666


 REPORTE: PUNT_FINAL
              precision    recall  f1-score   support

                   0.91      0.96      0.93    696764
           ,       0.16      0.04      0.06     25366
   

In [ ]:
import numpy as np
import os
import gc
import joblib
from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# --- 1. CONFIGURACIÓN ---
DIR_BASE = 'D:/Proyecto_Embeddings'

FILE_X_TRAIN = os.path.join(DIR_BASE, 'dataset_train_completo.dat')
FILE_X_EVAL  = os.path.join(DIR_BASE, 'dataset_eval_completo.dat')

FILES_Y = {
    'train': {
        'cap': os.path.join(DIR_BASE, 'y_train_cap.npy'),
        'ini': os.path.join(DIR_BASE, 'y_train_ini.npy'),
        'fin': os.path.join(DIR_BASE, 'y_train_fin.npy')
    },
    'eval': {
        'cap': os.path.join(DIR_BASE, 'y_eval_cap.npy'),
        'ini': os.path.join(DIR_BASE, 'y_eval_ini.npy'),
        'fin': os.path.join(DIR_BASE, 'y_eval_fin.npy')
    }
}

COLS = 772
DTYPE = np.float16

# --- CAMBIO CRÍTICO: BAJAMOS A 500.000 ---
# 500k filas pesan 1.4 GB. Esto entra seguro en cualquier hueco de RAM.
SAMPLES_FOR_TRAINING = 1_000_000

# --- 2. CONEXIÓN A DISCO ---
def get_rows(filepath, cols, dtype):
    return os.path.getsize(filepath) // (cols * np.dtype(dtype).itemsize)

print("🔌 Conectando datos...")
n_train = get_rows(FILE_X_TRAIN, COLS, DTYPE)
n_eval  = get_rows(FILE_X_EVAL, COLS, DTYPE)

X_train_disk = np.memmap(FILE_X_TRAIN, dtype=DTYPE, mode='r', shape=(n_train, COLS))
X_eval_disk  = np.memmap(FILE_X_EVAL,  dtype=DTYPE, mode='r', shape=(n_eval, COLS))

print("📥 Cargando etiquetas...")
y_train = {
    'capitalizacion': np.load(FILES_Y['train']['cap']),
    'punt_inicial':   np.load(FILES_Y['train']['ini']),
    'punt_final':     np.load(FILES_Y['train']['fin'])
}
y_eval = {
    'capitalizacion': np.load(FILES_Y['eval']['cap']),
    'punt_inicial':   np.load(FILES_Y['eval']['ini']),
    'punt_final':     np.load(FILES_Y['eval']['fin'])
}

# --- 3. SELECCIÓN DE LA MUESTRA (500k) ---
print(f"\n🎲 Seleccionando {SAMPLES_FOR_TRAINING:,} filas aleatorias...")

# Forzamos limpieza antes de pedir RAM
gc.collect()

np.random.seed(42)
indices_subset = np.random.choice(n_train, SAMPLES_FOR_TRAINING, replace=False)

print("   -> Reservando espacio en RAM (1.4 GB)...")
try:
    X_subset = np.empty((SAMPLES_FOR_TRAINING, COLS), dtype=np.float32)
except MemoryError:
    print("❌ ERROR FATAL: Ni siquiera hay 1.4GB continuos.")
    print("Debes reiniciar el kernel (Kernel -> Restart) y correr SOLO esta celda.")
    raise

print("   -> Llenando RAM por pedacitos...")
CHUNK_SIZE = 50000

for i in tqdm(range(0, SAMPLES_FOR_TRAINING, CHUNK_SIZE), desc="Cargando Subset"):
    end = min(i + CHUNK_SIZE, SAMPLES_FOR_TRAINING)
    indices_batch = indices_subset[i:end]
    X_subset[i:end] = X_train_disk[indices_batch].astype(np.float32)

print(f"✅ Subset cargado. Tamaño: {X_subset.nbytes / (1024**3):.2f} GB.")

# Liberamos el puntero al disco gigante
del X_train_disk
gc.collect()

# --- 4. BUCLE DE ENTRENAMIENTO (3 MODELOS) ---
modelos = {}
os.makedirs('modelos_rf', exist_ok=True)

targets_nombres = ['capitalizacion', 'punt_inicial', 'punt_final']

for nombre in targets_nombres:
    print(f"\n{'='*50}")
    print(f"🌲 ENTRENANDO RANDOM FOREST: {nombre.upper()}")
    print(f"{'='*50}")

    y_subset = y_train[nombre][indices_subset]

    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        class_weight='balanced',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )

    rf.fit(X_subset, y_subset)
    modelos[nombre] = rf

    joblib.dump(rf, f'modelos_rf/rf_{nombre}.pkl')
    print(f"💾 Guardado: modelos_rf/rf_{nombre}.pkl")

    # Evaluación
    print(f"📊 Evaluando en Set de Validación...")
    preds = []
    BATCH_PRED = 50000

    for i in tqdm(range(0, n_eval, BATCH_PRED), desc="Predicting"):
        end = min(i + BATCH_PRED, n_eval)
        batch_eval = X_eval_disk[i:end].astype(np.float32)
        p = rf.predict(batch_eval)
        preds.extend(p)

    print(f"\n--- REPORTE: {nombre.upper()} ---")
    print(classification_report(y_eval[nombre], preds))

    del preds
    gc.collect()

print("\n🎉 ¡PROCESO TERMINADO EXITOSAMENTE!")

🔌 Conectando datos...
📥 Cargando etiquetas...

🎲 Seleccionando 1,000,000 filas aleatorias...
   -> Reservando espacio en RAM (1.4 GB)...
   -> Llenando RAM por pedacitos...


Cargando Subset:   0%|          | 0/20 [00:00<?, ?it/s]

✅ Subset cargado. Tamaño: 2.88 GB.

🌲 ENTRENANDO RANDOM FOREST: CAPITALIZACION


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  4.3min finished


💾 Guardado: modelos_rf/rf_capitalizacion.pkl
📊 Evaluando en Set de Validación...


...

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  3.8min finished


💾 Guardado: modelos_rf/rf_punt_final.pkl
📊 Evaluando en Set de Validación...


Predicting:   0%|          | 0/16 [00:00<?, ?it/s]

[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.2s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.2s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.2s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0


--- REPORTE: PUNT_FINAL ---
              precision    recall  f1-score   support

                   0.97      0.81      0.89    696764
           ,       0.15      0.34      0.21     25366
           .       0.26      0.53      0.35     51874
           ?       0.06      0.32      0.10      6662

    accuracy                           0.78    780666
   macro avg       0.36      0.50      0.39    780666
weighted avg       0.89      0.78      0.82    780666


🎉 ¡PROCESO TERMINADO EXITOSAMENTE!


In [ ]:
df_forest.head()

,token,id,y_puntuacion_inicial,y_puntuacion_final,y_capitalizacion,n_word_in_sentence,sentence_length,relative_position_in_sentence,cosine_distance_next_token,cosine_distance_prev_token,is_subtoken,is_nan_cosine_distance_next_token,is_nan_cosine_distance_prev_token
0,él,14370,,,1,0,7,0.000,0.899363,-1.000000,False,False,True
1,no,10192,,,0,1,7,0.125,0.944345,0.899363,False,False,False
2,viene,12266,,,0,2,7,0.250,0.929369,0.944345,False,False,False
3,al,10164,,,0,3,7,0.375,0.980138,0.929369,False,False,False
4,trabajo,18100,,,0,4,7,0.500,0.839328,0.980138,False,False,False


## Separamos Train y Test data

In [ ]:
X = df_forest.drop(columns=['y_puntuacion_inicial', 'y_puntuacion_final', 'y_capitalizacion', 'token'])
y_ini = df_forest['y_puntuacion_inicial']
y_fin = df_forest['y_puntuacion_final']
y_cap = df_forest['y_capitalizacion']

In [ ]:
X_train, X_test, y_ini_train, y_ini_test, y_fin_train, y_fin_test, y_cap_train, y_cap_test = train_test_split(
    X, y_ini, y_fin, y_cap, test_size=0.2, random_state=33)
# Es necesario estratificar? Como lo hacemos? Tenemos que partir en held-out y desarrollo? y despues en train y test??

In [ ]:
# Entrenamos modelo para puntuación inicial
rf_ini = RandomForestClassifier(
  n_estimators=100,
  random_state=33,
  n_jobs=-1,
  class_weight='balanced'
)

rf_ini.fit(X_train, y_ini_train)
y_ini_pred = rf_ini.predict(X_test)
print(classification_report(y_ini_test, y_ini_pred))


              precision    recall  f1-score   support

                   0.99      1.00      1.00    127664
           ¿       0.71      0.35      0.46      1127

    accuracy                           0.99    128791
   macro avg       0.85      0.67      0.73    128791
weighted avg       0.99      0.99      0.99    128791



In [ ]:
# Entrenamos modelo para puntuación final
rf_fin = RandomForestClassifier(n_estimators=100, random_state=33, n_jobs=-1, class_weight='balanced')
rf_fin.fit(X_train, y_fin_train)
y_fin_pred = rf_fin.predict(X_test)
print(classification_report(y_fin_test, y_fin_pred))

              precision    recall  f1-score   support

                   0.96      1.00      0.98    114962
           ,       0.52      0.02      0.04      4149
           .       0.84      0.94      0.89      8573
           ?       0.17      0.04      0.07      1107

    accuracy                           0.95    128791
   macro avg       0.62      0.50      0.49    128791
weighted avg       0.93      0.95      0.94    128791



In [ ]:
# Entrenamos modelo para capitalizacion
rf_cap = RandomForestClassifier(n_estimators=100, random_state=33, n_jobs=-1, class_weight='balanced')
rf_cap.fit(X_train, y_cap_train)
y_cap_pred = rf_cap.predict(X_test)
print(classification_report(y_cap_test, y_cap_pred))

              precision    recall  f1-score   support

           0       0.94      0.99      0.97    110712
           1       0.94      0.65      0.77     17104
           2       0.92      0.15      0.26       630
           3       0.96      0.21      0.35       345

    accuracy                           0.94    128791
   macro avg       0.94      0.50      0.59    128791
weighted avg       0.94      0.94      0.94    128791



En todos los casos hay precision y recall altos unicamente para la clase mayoritaria (clase 0), o para las dos clases con mas ejemplos para los problemas multiclase. Esto puede deberse a desbalance de clases (la mayoria de los ejemplos son negativos). Si la precision es alta, la recall es baja o visceversa.
Mirar la accuracy seria engañoso: gran proporcion de las instancias bien clasificadas corresponden solo a la clase 0.